# Data Collection Notebook

This notebook version of `data_collection.py` keeps the same dataset-generation logic while splitting it into smaller sections that are easier to inspect, modify, and run incrementally.

It includes:

- PyTorch Geometric `.pt` export
- episode-level train/val/test splits
- tapered temporal sampling
- rollout metadata per graph
- a JSON sidecar describing the dataset recipe


In [12]:
import json
import os
from typing import Literal

import numpy as np
import pybullet as p
import torch
from scipy.optimize import linear_sum_assignment
from torch_geometric.data import Data, HeteroData, InMemoryDataset

from PyFlyt.core import Aviary

## Global Constants

This cell defines formation labels and split-specific seed offsets. The split offsets make validation and test episodes use different random streams from training even when you start from one base seed.


In [13]:
FORMATION_NAMES = ("a", "rectangle", "triangle", "random_cloud")
FORMATION_TO_ID = {name: idx for idx, name in enumerate(FORMATION_NAMES)}
SPLIT_NAMES = ("train", "val", "test")
SPLIT_SEED_OFFSETS = {"train": 0, "val": 1_000_000, "test": 2_000_000}
TASK_TYPES = ("setpoint_prediction", "residual_correction", "formation_assignment_homo", "formation_assignment_hetero")

## Wind and Episode Initialization

These helpers generate a simple wind field and sample randomized initial conditions for each episode. The initialization function accepts a NumPy random generator so each episode can be reproduced from a seed.


In [14]:
def wind_generator(time: float, position: np.ndarray):
    """Generates an upward draft with random turbulence."""
    wind = np.zeros_like(position)
    wind[:, 2] = np.sin(time) * 0.5 + np.random.normal(0, 0.2, size=(len(position),))
    wind[:, 0] = np.cos(time / 2.0) * 0.3
    wind[:, 1] = np.sin(time / 2.0) * 0.3
    return wind


def sample_episode_initial_conditions(
    num_drones: int,
    rng: np.random.Generator,
    xy_limit: float = 10.0,
    altitude_range: tuple[float, float] = (0.5, 5.0),
):
    start_pos = rng.uniform(-xy_limit, xy_limit, size=(num_drones, 3))
    start_pos[:, 2] = rng.uniform(altitude_range[0], altitude_range[1], size=(num_drones,))

    start_orn = np.zeros((num_drones, 3))
    start_orn[:, 2] = rng.uniform(-np.pi, np.pi, size=(num_drones,))
    return start_pos, start_orn

def sample_obstacles(
    rng: np.random.Generator,
    start_positions: np.ndarray,
    max_obstacles: int = 25,
    xy_bounds: tuple[float, float] = (-15.0, 15.0),
    radius_range: tuple[float, float] = (0.3, 2.0),
    drone_clearance: float = 3.0,
    obstacle_clearance: float = 1.0,
    max_attempts_per_obstacle: int = 100,
) -> np.ndarray:
    """
    Generate randomized obstacles with rejection sampling.

    Returns np.ndarray of shape (N, 3): columns are [x, y, radius].
    N is random per call (0 to max_obstacles).
    Obstacles are guaranteed to not overlap drone start positions.
    """
    num_obstacles = int(rng.integers(0, max_obstacles + 1))
    if num_obstacles == 0:
        return np.zeros((0, 3))

    # Protected zones: drone start XY positions
    protected_xy = start_positions[:, :2] if len(start_positions) > 0 else np.zeros((0, 2))

    accepted: list[list[float]] = []

    for _ in range(num_obstacles):
        for _ in range(max_attempts_per_obstacle):
            cx = rng.uniform(xy_bounds[0], xy_bounds[1])
            cy = rng.uniform(xy_bounds[0], xy_bounds[1])
            cr = rng.uniform(radius_range[0], radius_range[1])

            # Check 1: Clear of all drone start positions
            if len(protected_xy) > 0:
                dists_to_protected = np.linalg.norm(
                    protected_xy - np.array([cx, cy]), axis=1
                )
                if np.any(dists_to_protected < drone_clearance + cr):
                    continue

            # Check 2: Clear of previously placed obstacles (surface-to-surface)
            if len(accepted) > 0:
                existing = np.array(accepted)
                dists_to_existing = np.linalg.norm(
                    existing[:, :2] - np.array([cx, cy]), axis=1
                )
                min_surface_dist = dists_to_existing - existing[:, 2] - cr
                if np.any(min_surface_dist < obstacle_clearance):
                    continue

            accepted.append([cx, cy, cr])
            break

    if len(accepted) == 0:
        return np.zeros((0, 3))
    return np.array(accepted, dtype=np.float64)

def resolve_formation_name(dataset_type: str):
    if dataset_type in {"a", "formation_a"}:
        return "a"
    if dataset_type in {"rectangle", "rectangular", "formation_rectangle"}:
        return "rectangle"
    if dataset_type in {"triangle", "formation_triangle"}:
        return "triangle"
    if dataset_type in {"random_cloud", "formation_random_cloud"}:
        return "random_cloud"
    return None

## Formation Geometry

These helpers define how each formation is laid out in the XY plane. The setpoints are centered around the episode's mean initial position so the swarm moves into formation without teleporting the target far away from the sampled scenario.


In [15]:
def _formation_a_offsets(num_drones: int, spacing: float = 2.0):
    offsets = np.zeros((num_drones, 3))
    if num_drones == 1:
        return offsets
        
    num_crossbar = (num_drones // 5) if num_drones > 5 else 0
    num_v_legs = num_drones - num_crossbar
    
    if num_v_legs % 2 == 0:
            num_crossbar += 1
            num_v_legs -= 1
        
    
    offsets[0] = np.array([0.0, 0.0, 0.0])
    for idx in range(1, num_v_legs):
        level = (idx + 1) // 2
        side = -1.0 if idx % 2 == 1 else 1.0 
        offsets[idx, 0] = side * level * spacing 
        offsets[idx, 1] = level * spacing
        
    if num_crossbar > 0:
        max_level = (num_v_legs + 1) // 2
        mid_level = max(1, max_level // 2)
        
        left_x = -mid_level * spacing
        right_x = mid_level * spacing
        width = right_x - left_x
        
        for crossbar_idx in range(num_crossbar):
            fraction = (crossbar_idx + 1) / (num_crossbar + 1)
            target_idx = num_v_legs + crossbar_idx
            offsets[target_idx, 0] = left_x + (fraction * width)
            offsets[target_idx, 1] = mid_level * spacing

    offsets[:, :2] -= np.mean(offsets[:, :2], axis=0) 
    return offsets


def _formation_rectangle_offsets(num_drones: int, spacing: float = 2.0):
    offsets = np.zeros((num_drones, 3))
    if num_drones == 0:
        return offsets

    # Base shape on required perimeter
    perimeter = num_drones * spacing
    side = perimeter / 4.0
    
    # 4 Corners (bottom-left, bottom-right, top-right, top-left)
    corners = [
        np.array([-side/2, -side/2]),
        np.array([ side/2, -side/2]),
        np.array([ side/2,  side/2]),
        np.array([-side/2,  side/2])
    ]
    
    # 1. Fill corners first
    for i in range(min(num_drones, 4)):
        offsets[i, :2] = corners[i]
        
    # 2. Distribute remaining drones on the borders
    if num_drones > 4:
        remaining = num_drones - 4
        # Evenly divide remaining among the 4 edges
        drones_per_edge = [remaining // 4 + (1 if e < remaining % 4 else 0) for e in range(4)]
        
        current_idx = 4
        for e in range(4):
            start_c = corners[e]
            end_c = corners[(e + 1) % 4]
            num_edge = drones_per_edge[e]
            
            for step in range(1, num_edge + 1):
                fraction = step / (num_edge + 1)
                offsets[current_idx, :2] = start_c + fraction * (end_c - start_c)
                current_idx += 1
                
    offsets[:, :2] -= np.mean(offsets[:, :2], axis=0)
    return offsets


def _formation_triangle_offsets(num_drones: int, spacing: float = 2.0):
    offsets = np.zeros((num_drones, 3))
    if num_drones == 0:
        return offsets

    # Base shape on required perimeter
    perimeter = num_drones * spacing
    side = perimeter / 3.0
    h = side * np.sqrt(3) / 2.0
    
    # 3 Corners (bottom-left, bottom-right, top-center)
    corners = [
        np.array([-side/2, -h/3]),
        np.array([ side/2, -h/3]),
        np.array([      0, 2*h/3])
    ]
    
    # 1. Fill corners first
    for i in range(min(num_drones, 3)):
        offsets[i, :2] = corners[i]
        
    # 2. Distribute remaining drones on the borders
    if num_drones > 3:
        remaining = num_drones - 3
        # Evenly divide remaining among the 3 edges
        drones_per_edge = [remaining // 3 + (1 if e < remaining % 3 else 0) for e in range(3)]
        
        current_idx = 3
        for e in range(3):
            start_c = corners[e]
            end_c = corners[(e + 1) % 3]
            num_edge = drones_per_edge[e]
            
            for step in range(1, num_edge + 1):
                fraction = step / (num_edge + 1)
                offsets[current_idx, :2] = start_c + fraction * (end_c - start_c)
                current_idx += 1
                
    offsets[:, :2] -= np.mean(offsets[:, :2], axis=0)
    return offsets


def apply_obstacle_avoidance(slots, obstacles, padding=1.0):
    """Pushes formation slots out of obstacles if they overlap.
    
    obstacles: (N, 3) array with columns [x, y, radius].
    """
    if len(obstacles) == 0:
        return slots
    
    safe_slots = np.copy(slots)
    
    for i in range(len(safe_slots)):
        for obs in obstacles:
            obs_center = obs[:2]
            obs_r = obs[2]
            min_dist = obs_r + padding
            diff = safe_slots[i, :2] - obs_center
            dist = np.linalg.norm(diff)
            if dist < min_dist:
                direction = diff / (dist + 1e-6)
                safe_slots[i, :2] = obs_center + direction * min_dist
                
    return safe_slots


def _build_formation_setpoints(formation_name: str, start_pos: np.ndarray, obstacles=np.empty((0, 3))):
    num_drones = len(start_pos)
    formation_center = np.mean(start_pos[:, :2], axis=0)
    target_altitude = np.mean(start_pos[:, 2])

    if formation_name == "a":
        offsets = _formation_a_offsets(num_drones)
    elif formation_name == "rectangle":
        offsets = _formation_rectangle_offsets(num_drones)
    elif formation_name == "triangle":
        offsets = _formation_triangle_offsets(num_drones)
    else:
        return None, None, None

    # Step 1: Base slots (naive)
    global_slots = np.zeros((num_drones, 3))
    global_slots[:, :2] = formation_center + offsets[:, :2]
    
    # Step 2: Apply obstacle avoidance to shift slots dynamically
    safe_global_slots = apply_obstacle_avoidance(global_slots, obstacles, padding=1.0)
    
    # Step 3: Optimal Assignment
    dist_matrix = np.linalg.norm(start_pos[:, None, :2] - safe_global_slots[None, :, :2], axis=2)
    _, assigned_indices = linear_sum_assignment(dist_matrix)
    
    # Step 4: Build final setpoints
    setpoints = np.zeros((num_drones, 4))
    for i in range(num_drones):
        slot_idx = assigned_indices[i]
        setpoints[i, :2] = safe_global_slots[slot_idx, :2]
        setpoints[i, 2] = 0.0 # Yaw
        setpoints[i, 3] = target_altitude

    return setpoints, assigned_indices, offsets

## Random Cloud Setpoint Generation

Generates randomized 3D setpoints using **rejection sampling** with a minimum
safe distance of 1.5 m between any two targets. This prevents the GNN from
memorising specific trajectory shapes (distributional shift).

50% of training episodes use Random Cloud targets instead of structured formations.


In [ ]:
# ==============================================================================
# Random Cloud Setpoint Generation  (Rejection Sampling)
# ==============================================================================

def generate_random_cloud_setpoints(
    num_drones: int,
    rng: np.random.Generator,
    x_bounds: tuple[float, float] = (-10.0, 10.0),  # match spawn bounds
    y_bounds: tuple[float, float] = (-10.0, 10.0),  # match spawn bounds
    z_bounds: tuple[float, float] = (1.0, 9.5),   # 0.5m ceiling brake margin
    min_dist: float = 1.5,
    max_attempts: int = 5_000,
) -> np.ndarray:
    """
    Generate N random 3-D setpoints via rejection sampling.

    Ensures every accepted point is at least `min_dist` metres from all
    previously accepted points (3-D Euclidean distance).
    """
    accepted: list[np.ndarray] = []
    attempts = 0

    while len(accepted) < num_drones and attempts < max_attempts:
        attempts += 1
        candidate = np.array(
            [
                rng.uniform(x_bounds[0], x_bounds[1]),
                rng.uniform(y_bounds[0], y_bounds[1]),
                rng.uniform(z_bounds[0], z_bounds[1]),
            ],
            dtype=np.float64,
        )
        if len(accepted) == 0:
            accepted.append(candidate)
            continue
        existing = np.array(accepted)
        dists = np.linalg.norm(existing - candidate, axis=1)
        if np.all(dists >= min_dist):
            accepted.append(candidate)

    if len(accepted) < num_drones:
        raise RuntimeError(
            f"Could only place {len(accepted)}/{num_drones} setpoints "
            f"with min_dist={min_dist} m after {max_attempts} attempts. "
            f"Reduce min_dist or expand the bounding box."
        )

    return np.array(accepted, dtype=np.float64)  # (N, 3)


# ==============================================================================
# Pipeline wrapper: Random Cloud -> same interface as _build_formation_setpoints
# ==============================================================================

def _build_random_cloud_setpoints(
    start_pos: np.ndarray,
    rng: np.random.Generator,
    x_bounds: tuple[float, float] = (-10.0, 10.0),  # match spawn bounds
    y_bounds: tuple[float, float] = (-10.0, 10.0),  # match spawn bounds
    z_bounds: tuple[float, float] = (1.0, 9.5),   # 0.5m ceiling brake margin
    min_dist: float = 1.5,
    obstacles: np.ndarray = np.empty((0, 3)),
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Build random-cloud setpoints compatible with the existing pipeline.

    Returns setpoints (N,4), assigned_indices (N,), offsets (N,3).
    """
    num_drones = len(start_pos)
    target_positions = generate_random_cloud_setpoints(
        num_drones, rng, x_bounds, y_bounds, z_bounds, min_dist
    )
    global_slots = np.zeros((num_drones, 3))
    global_slots[:, :2] = target_positions[:, :2]
    safe_slots = apply_obstacle_avoidance(
        global_slots, obstacles, padding=1.0
    )
    dist_matrix = np.linalg.norm(
        start_pos[:, None, :2] - safe_slots[None, :, :2], axis=2
    )
    _, assigned_indices = linear_sum_assignment(dist_matrix)

    setpoints = np.zeros((num_drones, 4))
    for i in range(num_drones):
        slot_idx = assigned_indices[i]
        setpoints[i, 0] = safe_slots[slot_idx, 0]
        setpoints[i, 1] = safe_slots[slot_idx, 1]
        setpoints[i, 2] = rng.uniform(-np.pi, np.pi)
        setpoints[i, 3] = target_positions[slot_idx, 2]

    offsets = np.zeros((num_drones, 3))
    return setpoints, assigned_indices, offsets


## Environment Creation and Setpoint Generation

This part creates the PyFlyt `Aviary`, enables optional wind, and generates setpoints for either a formation task or a generic random-hovering style task.


In [17]:
def create_aviary(
    start_pos: np.ndarray,
    start_orn: np.ndarray,
    environmental_wind: bool,
    obstacles: np.ndarray = np.empty((0, 3)),
    graphical: bool = False,
):
    env = Aviary(
        start_pos=start_pos,
        start_orn=start_orn,
        drone_type="quadx",
        render=graphical,
    )

    if environmental_wind:
        env.register_wind_field_function(wind_generator)

    env.set_mode(7)

    # Spawn simple cylinder obstacles if present
    if len(obstacles) > 0:
        physics_client = env._client
        for obs in obstacles:
            obs_x, obs_y, obs_r = obs[0], obs[1], obs[2]
            col_id = p.createCollisionShape(p.GEOM_CYLINDER, radius=obs_r, height=10.0, physicsClientId=physics_client)
            vis_id = p.createVisualShape(p.GEOM_CYLINDER, radius=obs_r, length=10.0, rgbaColor=[1,0,0,0.5], physicsClientId=physics_client)
            p.createMultiBody(baseMass=0, baseCollisionShapeIndex=col_id, baseVisualShapeIndex=vis_id, basePosition=[obs_x, obs_y, 5.0], physicsClientId=physics_client)
        env.register_all_new_bodies()

    return env


def build_setpoints(
    dataset_type: str,
    start_pos: np.ndarray,
    start_orn: np.ndarray,
    rng: np.random.Generator,
    obstacles: np.ndarray = np.empty((0, 3)),
):
    num_drones = len(start_pos)

    # ── Random Cloud branch ───────────────────────────────────────────────────
    # Rejection-sampled random targets scattered inside a 10×10×9 m flight
    # space with a minimum safe distance of 1.5 m between any two setpoints.
    # This branch handles dataset_type in {"random_cloud", "cloud"}.
    if dataset_type in {"random_cloud", "cloud"}:
        return _build_random_cloud_setpoints(
            start_pos, rng,
            obstacles=obstacles,
        )

    formation_name = resolve_formation_name(dataset_type)
    
    if formation_name is not None:
        setpoints, col_ind, offsets = _build_formation_setpoints(formation_name, start_pos, obstacles)
        if setpoints is not None:
            return setpoints, col_ind, offsets

    # Default fallback (hovering or aggressive target changing)
    setpoints = np.zeros((num_drones, 4))
    if dataset_type == "hovering":
        setpoints[:, :2] = start_pos[:, :2]
        setpoints[:, 2] = start_orn[:, 2]
        setpoints[:, 3] = start_pos[:, 2]
    else:
        radius = 10.0 if dataset_type == "aggressive" else 5.0
        setpoints[:, :2] = start_pos[:, :2] + rng.uniform(-radius, radius, size=(num_drones, 2))
        setpoints[:, 2] = rng.uniform(-np.pi, np.pi, size=(num_drones,))
        setpoints[:, 3] = rng.uniform(1.0, radius, size=(num_drones,))

    col_ind = np.arange(num_drones)
    return setpoints, col_ind, np.zeros((num_drones, 3))

## State Processing and Graph Construction

These helpers transform raw drone state into graph node features, target vectors, and communication edges. This is the part that turns the simulator rollout into supervised learning samples for the GNN.


In [18]:
def maybe_add_sensor_noise(
    global_pos: np.ndarray,
    global_euler: np.ndarray,
    local_lin_vel: np.ndarray,
    local_ang_vel: np.ndarray,
    noisy_sensors: bool,
    noise_variance: float,
):
    if not noisy_sensors:
        return global_pos, global_euler, local_lin_vel, local_ang_vel

    global_pos = global_pos + np.random.normal(0, noise_variance, size=3)
    global_euler = global_euler + np.random.normal(0, noise_variance, size=3)
    local_lin_vel = local_lin_vel + np.random.normal(0, noise_variance, size=3)
    local_ang_vel = local_ang_vel + np.random.normal(0, noise_variance, size=3)
    return global_pos, global_euler, local_lin_vel, local_ang_vel


def build_drone_features(
    drone,
    drone_idx: int,
    setpoint: np.ndarray,
    assigned_slot_idx: int,
    naive_offset: np.ndarray,
    _task_type: str,
    noisy_sensors: bool,
    noise_variance: float,
    formation_one_hot: np.ndarray | None,
    obstacles: np.ndarray,
    include_formation_in_state: bool,
    start_pos_center: np.ndarray,
    next_step_label: np.ndarray | None,
    state_override: tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray] | None = None,
):
    if state_override is None:
        state = drone.state
        global_pos = np.array(state[3], copy=True)
        global_euler = np.array(state[1], copy=True)  # global euler angles
        local_lin_vel = np.array(state[2], copy=True)
        local_ang_vel = np.array(state[0], copy=True)
    else:
        global_pos, global_euler, local_lin_vel, local_ang_vel = state_override
        global_pos = np.array(global_pos, copy=True)
        global_euler = np.array(global_euler, copy=True)
        local_lin_vel = np.array(local_lin_vel, copy=True)
        local_ang_vel = np.array(local_ang_vel, copy=True)

    global_pos, global_euler, local_lin_vel, local_ang_vel = maybe_add_sensor_noise(
        global_pos,
        global_euler,
        local_lin_vel,
        local_ang_vel,
        noisy_sensors,
        noise_variance,
    )

    rotation_quaternion = p.getQuaternionFromEuler(global_euler)
    rot_matrix = np.array(p.getMatrixFromQuaternion(rotation_quaternion)).reshape(3, 3)
    global_lin_vel = rot_matrix @ local_lin_vel

    # Pseudo-LiDAR raycast for obstacle detection
    num_rays = 16
    max_range = 5.0
    obs_features = np.full(num_rays, max_range)
    if len(obstacles) > 0:
        drone_pos = global_pos[:2]
        drone_yaw = global_euler[2]
        angles = np.linspace(0, 2 * np.pi, num_rays, endpoint=False) + drone_yaw
        rays = np.stack([np.cos(angles), np.sin(angles)], axis=1)

        obs_centers = obstacles[:, :2]
        obs_radii = obstacles[:, 2]

        for i, ray in enumerate(rays):
            W = obs_centers - drone_pos
            t = np.dot(W, ray)
            hit_mask = t > 0
            if np.any(hit_mask):
                W_hit = W[hit_mask]
                t_hit = t[hit_mask]
                r_hit = obs_radii[hit_mask]
                d_sq = np.sum(W_hit**2, axis=1) - t_hit**2
                valid_hit = d_sq <= r_hit**2
                if np.any(valid_hit):
                    hit_dists = t_hit[valid_hit] - np.sqrt(r_hit[valid_hit]**2 - d_sq[valid_hit])
                    valid_dists = hit_dists[hit_dists > 0]
                    if len(valid_dists) > 0:
                        obs_features[i] = min(max_range, np.min(valid_dists))

    target_global_pos = np.array([setpoint[0], setpoint[1], setpoint[3]])
    target_global_yaw = setpoint[2]
    global_pos_error = target_global_pos - global_pos
    local_pos_error = rot_matrix.T @ global_pos_error
    yaw_error = target_global_yaw - global_euler[2]
    yaw_error = (yaw_error + np.pi) % (2 * np.pi) - np.pi

    # Boundary awareness features (Fix 3)
    distance_to_floor = global_pos[2]
    distance_to_ceiling = 10.0 - global_pos[2]

    gnn_input_state = np.concatenate([
        local_lin_vel, local_ang_vel, obs_features, local_pos_error, [yaw_error],
        [distance_to_floor, distance_to_ceiling]
    ])
    if include_formation_in_state and formation_one_hot is not None:
        gnn_input_state = np.concatenate([gnn_input_state, formation_one_hot])

    if next_step_label is None:
        raise ValueError("next_step_label must be provided to build displacement targets.")
    gnn_input_target = np.array(next_step_label, copy=True)

    return gnn_input_state, gnn_input_target, global_pos, global_euler, global_lin_vel


def build_edges(
    global_positions: np.ndarray,
    global_lin_vels: np.ndarray,
    global_eulers: np.ndarray,
    communication_radius: float,
):
    edges = []
    edge_attrs = []
    num_drones = len(global_positions)

    for i in range(num_drones):
        rot_quat = p.getQuaternionFromEuler(global_eulers[i])
        rot_matrix = np.array(p.getMatrixFromQuaternion(rot_quat)).reshape(3, 3)
        for j in range(num_drones):
            if i == j:
                continue

            rel_pos_world = global_positions[j] - global_positions[i]
            dist = np.linalg.norm(rel_pos_world)
            if dist <= communication_radius:
                rel_vel_world = global_lin_vels[j] - global_lin_vels[i]
                rel_pos_local = rot_matrix.T @ rel_pos_world
                rel_vel_local = rot_matrix.T @ rel_vel_world
                edges.append([i, j])
                edge_attrs.append(np.concatenate([rel_pos_local, [dist], rel_vel_local]))

    return edges, edge_attrs


def collect_step_data(
    env,
    active_drones: list[int],
    setpoints,
    slot_assignments: list[int],
    naive_offsets: np.ndarray,
    task_type: str,
    noisy_sensors: bool,
    noise_variance: float,
    communication_radius: float,
    formation_one_hot: np.ndarray | None,
    obstacles: np.ndarray,
    include_formation_in_state: bool,
    start_pos_center: np.ndarray,
    state_cache: list[tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]] | None = None,
    next_step_labels: list[np.ndarray] | None = None,
):
    episode_states = []
    episode_targets = []
    global_positions = []
    global_eulers = []
    global_lin_vels = []

    for i, drone_idx in enumerate(active_drones):
        drone = env.drones[drone_idx]
        slot_idx = slot_assignments[i]
        state_override = None
        if state_cache is not None:
            state_override = state_cache[i]
        next_step_label = None
        if next_step_labels is not None:
            next_step_label = next_step_labels[i]
        gnn_input_state, gnn_input_target, global_pos, global_euler, global_lin_vel = build_drone_features(
            drone,
            drone_idx,
            setpoints[i],
            slot_idx,
            naive_offsets[slot_idx],
            task_type,
            noisy_sensors,
            noise_variance,
            formation_one_hot,
            obstacles,
            include_formation_in_state,
            start_pos_center,
            next_step_label=next_step_label,
            state_override=state_override,
        )
        episode_states.append(gnn_input_state)
        episode_targets.append(gnn_input_target)
        global_positions.append(global_pos)
        global_eulers.append(global_euler)
        global_lin_vels.append(global_lin_vel)

    edges, edge_attrs = build_edges(
        np.array(global_positions),
        np.array(global_lin_vels),
        np.array(global_eulers),
        communication_radius,
    )
    return episode_states, episode_targets, edges, edge_attrs, global_positions, global_eulers


def build_next_step_labels(
    env,
    active_drones: list[int],
    current_positions: list[np.ndarray],
    current_eulers: list[np.ndarray],
):
    # Labels are displacement in the current body frame plus yaw delta.
    labels = []
    for i, drone_idx in enumerate(active_drones):
        next_pos = np.array(env.drones[drone_idx].state[3], copy=True)
        next_euler = np.array(env.drones[drone_idx].state[1], copy=True)
        displacement = next_pos - current_positions[i]
        current_rotation = p.getQuaternionFromEuler(current_eulers[i])
        current_rot_matrix = np.array(p.getMatrixFromQuaternion(current_rotation)).reshape(3, 3)
        local_displacement = current_rot_matrix.T @ displacement
        yaw_delta = next_euler[2] - current_eulers[i][2]
        yaw_delta = (yaw_delta + np.pi) % (2 * np.pi) - np.pi
        label = np.concatenate([local_displacement, [yaw_delta]]).astype(np.float32)
        labels.append(label)
    return labels

## Split Logic and Metadata

This section implements the practical dataset design choices:

- split by episode rather than by graph
- use separate seed streams for train, val, and test
- write a metadata sidecar so the dataset recipe is reproducible


In [19]:
def compute_split_episode_counts(
    num_episodes: int,
    split_ratios: tuple[float, float, float],
):
    if len(split_ratios) != 3:
        raise ValueError("split_ratios must contain exactly three values for train, val, test.")
    if not np.isclose(sum(split_ratios), 1.0):
        raise ValueError("split_ratios must sum to 1.0.")

    counts = [int(num_episodes * ratio) for ratio in split_ratios]
    remainder = num_episodes - sum(counts)
    for idx in range(remainder):
        counts[idx] += 1

    return {split_name: count for split_name, count in zip(SPLIT_NAMES, counts)}


def resolve_split_spread_scale(
    split_name: str,
    validation_spread_scale: float,
    test_spread_scale: float,
):
    if split_name == "val":
        return validation_spread_scale
    if split_name == "test":
        return test_spread_scale
    return 1.0


def build_episode_seed(seed: int | None, split_name: str, split_episode_idx: int):
    if seed is None:
        return None
    return int(seed + SPLIT_SEED_OFFSETS[split_name] + split_episode_idx)


def write_dataset_metadata(metadata_path: str, metadata: dict):
    with open(metadata_path, "w", encoding="ascii") as metadata_file:
        json.dump(metadata, metadata_file, indent=2)
        metadata_file.write("\n")


def save_dataset(
    dataset_path: str,
    all_graphs,
    formation_names,
    split_name: str,
):
    data, slices = InMemoryDataset.collate(all_graphs)
    torch.save(
        {
            "data": data,
            "slices": slices,
            "formation_names": formation_names,
            "split_name": split_name,
        },
        dataset_path,
    )

## Main Dataset Generator

This is the full generation loop. It creates split-specific episodes, applies the tapered sampling policy, stores one PyG graph per retained step, and writes both split files and a metadata JSON file.


In [20]:
def generate_dataset(
    num_episodes: int = 300,
    max_steps: int = 300,
    dataset_name: str = "formation_dataset",
    dataset_type: str = "mixed_formations",
    task_type: Literal[
        "setpoint_prediction",
        "residual_correction",
        "formation_assignment_homo",
        "formation_assignment_hetero"
    ] = "setpoint_prediction",
    max_obstacles: int = 25,
    inject_failures: bool = False,
    dynamic_formation: bool = False,
    noisy_sensors: bool = False,
    noise_variance: float = 0.01,
    environmental_wind: bool = False,
    graphical: bool = False,
    communication_radius: float = np.inf,
    include_formation_in_state: bool = True,
    mixed_formation_types: tuple = FORMATION_NAMES,
    cloud_ratio: float = 0.5,
    split_ratios: tuple[float, float, float] = (0.8, 0.1, 0.1),
    seed: int = 12345,
    base_xy_limit: float = 10.0,
    altitude_range: tuple[float, float] = (0.5, 5.0),
    validation_spread_scale: float = 1.25,
    test_spread_scale: float = 1.5,
    save_interval: int = 10,          # Fix 1: strict 10-Hz modulo saving
    conv_stopping: bool = True,
    conv_threshold: float = 0.2,
) -> tuple[dict[str, str], str]:
    if '__file__' in globals():
        script_dir = os.path.dirname(os.path.abspath(__file__))
    else:
        script_dir = os.path.abspath(os.getcwd())
    repo_root = os.path.dirname(script_dir)
    datasets_dir = os.path.join(repo_root, "datasets")
    os.makedirs(datasets_dir, exist_ok=True)

    if task_type not in TASK_TYPES:
        raise ValueError(f"task_type must be one of {TASK_TYPES}")

    dataset_prefix = os.path.join(datasets_dir, f"{dataset_name}_{dataset_type}")
    split_episode_counts = compute_split_episode_counts(num_episodes, split_ratios)
    split_graphs = {split_name: [] for split_name in SPLIT_NAMES}
    split_summaries = {}
    episode_records = []

    global_episode_id = 0
    for split_name in SPLIT_NAMES:
        split_episode_count = split_episode_counts[split_name]
        split_spread_scale = resolve_split_spread_scale(
            split_name,
            validation_spread_scale,
            test_spread_scale,
        )
        split_summaries[split_name] = {
            "num_episodes": split_episode_count,
            "num_graphs": 0,
            "spread_scale": split_spread_scale,
        }

        for split_episode_idx in range(split_episode_count):
            episode_seed = build_episode_seed(seed, split_name, split_episode_idx)
            rng = np.random.default_rng(episode_seed)

            print(
                f"[{dataset_name}] Starting {split_name} episode {split_episode_idx + 1}/{split_episode_count} ..."
            )

            num_drones = int(rng.integers(10, 21))
            episode_dataset_type = dataset_type
            if dataset_type in {"mixed_formations", "mixed", "formations"}:
                if rng.random() < cloud_ratio:
                    episode_dataset_type = "random_cloud"
                else:
                    structured = [t for t in mixed_formation_types
                                  if t != "random_cloud"]
                    if structured:
                        episode_dataset_type = str(rng.choice(structured))
                    else:
                        episode_dataset_type = "random_cloud"

            xy_limit = base_xy_limit * split_spread_scale
            start_pos, start_orn = sample_episode_initial_conditions(
                num_drones, rng, xy_limit=xy_limit, altitude_range=altitude_range
            )
            obstacles = sample_obstacles(rng, start_pos, max_obstacles=max_obstacles)

            env = create_aviary(start_pos, start_orn, environmental_wind, obstacles, graphical)
            setpoints, col_ind, naive_offsets = build_setpoints(episode_dataset_type, start_pos, start_orn, rng, obstacles)
            active_drones = list(range(num_drones))

            formation_name = resolve_formation_name(episode_dataset_type)
            formation_id = -1
            if formation_name is not None:
                formation_id = FORMATION_TO_ID[formation_name]

            formation_one_hot = None
            if formation_id >= 0:
                formation_one_hot = np.zeros(len(FORMATION_NAMES), dtype=np.float32)
                formation_one_hot[formation_id] = 1.0

            for i, drone_idx in enumerate(active_drones):
                env.set_setpoint(drone_idx, setpoints[i])

            saved_steps = 0
            steps_since_last_event = 0
            failure_injected_this_episode = False

            # ── Fix 4: Frame-stacking memory (k=2) ─────────────────────────
            prev_drone_features: dict[int, np.ndarray] = {}

            for step_idx in range(max_steps):
                steps_since_last_event += 1

                # Optionally inject mid-flight failure
                should_inject_failure = inject_failures and not failure_injected_this_episode and step_idx >= max_steps // 2
                if should_inject_failure and len(active_drones) > 2:
                    failed_drone = active_drones.pop()
                    env.drones[failed_drone].set_mode(0)
                    active_start_pos = np.array([env.drones[idx].state[3] for idx in active_drones])
                    setpoints, col_ind, naive_offsets = build_setpoints(episode_dataset_type, active_start_pos, start_orn[:len(active_drones)], rng, obstacles)
                    for i, drone_idx in enumerate(active_drones):
                        env.drones[drone_idx].set_mode(7)
                        env.set_setpoint(drone_idx, setpoints[i])
                    failure_injected_this_episode = True
                    steps_since_last_event = 0
                    # Reset frame-stacking memory after topology change
                    prev_drone_features.clear()

                # Optionally rotate formation dynamically
                if dynamic_formation and step_idx > 0 and step_idx % 100 == 0:
                    next_shape = str(rng.choice(tuple(set(FORMATION_NAMES) - {formation_name})))
                    formation_name = next_shape
                    formation_id = FORMATION_TO_ID[formation_name]
                    formation_one_hot = np.zeros(len(FORMATION_NAMES), dtype=np.float32)
                    formation_one_hot[formation_id] = 1.0
                    active_start_pos = np.array([env.drones[idx].state[3] for idx in active_drones])
                    setpoints, col_ind, naive_offsets = build_setpoints(formation_name, active_start_pos, start_orn[:len(active_drones)], rng, obstacles)
                    for i, drone_idx in enumerate(active_drones):
                        env.set_setpoint(drone_idx, setpoints[i])
                    steps_since_last_event = 0
                    # Reset frame-stacking memory after formation change
                    prev_drone_features.clear()

                # ── Fix 1: Strict modulo saving (no convergence bypass) ─────
                should_save_step = (step_idx % save_interval == 0)

                state_cache = None
                current_positions = None
                current_eulers = None

                if should_save_step:
                    state_cache = []
                    current_positions = []
                    current_eulers = []
                    for drone_idx in active_drones:
                        state = env.drones[drone_idx].state
                        local_ang_vel = np.array(state[0], copy=True)
                        global_euler = np.array(state[1], copy=True)
                        local_lin_vel = np.array(state[2], copy=True)
                        global_pos = np.array(state[3], copy=True)
                        state_cache.append((global_pos, global_euler, local_lin_vel, local_ang_vel))
                        current_positions.append(global_pos)
                        current_eulers.append(global_euler)

                env.step()

                if should_save_step:
                    episode_labels = build_next_step_labels(env, active_drones, current_positions, current_eulers)
                    (
                        episode_states,
                        _,
                        edges,
                        edge_attrs,
                        global_positions,
                        global_eulers,
                    ) = collect_step_data(
                        env, active_drones, setpoints, col_ind, naive_offsets, task_type,
                        noisy_sensors, noise_variance, communication_radius, formation_one_hot,
                        obstacles, include_formation_in_state, np.mean(start_pos[:, :2], axis=0),
                        state_cache=state_cache,
                        next_step_labels=episode_labels,
                    )

                    # ── Fix 4: Frame stacking (k=2) ────────────────────────
                    stacked_states = []
                    for i, drone_idx in enumerate(active_drones):
                        current_feat = episode_states[i]
                        prev_feat = prev_drone_features.get(
                            drone_idx, np.zeros_like(current_feat)
                        )
                        stacked_states.append(
                            np.concatenate([current_feat, prev_feat])
                        )

                    # Update frame-stacking memory
                    for i, drone_idx in enumerate(active_drones):
                        prev_drone_features[drone_idx] = np.array(
                            episode_states[i], copy=True
                        )

                    x = torch.as_tensor(np.asarray(stacked_states), dtype=torch.float32)
                    target = torch.as_tensor(np.asarray(episode_labels), dtype=torch.float32)

                    if edges:
                        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
                        edge_attr = torch.as_tensor(np.asarray(edge_attrs), dtype=torch.float32)
                    else:
                        edge_index = torch.empty((2, 0), dtype=torch.long)
                        edge_attr = torch.empty((0, 7), dtype=torch.float32)

                    if task_type == "formation_assignment_hetero":
                        graph = HeteroData()
                        graph["drone"].x = x
                        graph["drone"].target = target
                        graph["drone"].pos = torch.as_tensor(np.asarray(global_positions), dtype=torch.float32)
                        graph["slot"].x = torch.as_tensor(naive_offsets, dtype=torch.float32)
                        graph["drone", "communicates", "drone"].edge_index = edge_index
                        graph["drone", "communicates", "drone"].edge_attr = edge_attr

                        drones = torch.arange(len(active_drones), dtype=torch.long)
                        slots = torch.as_tensor(np.asarray(col_ind), dtype=torch.long)
                        graph["drone", "assigned_to", "slot"].edge_label_index = torch.stack([drones, slots], dim=0)

                        graph.formation_id = torch.tensor([formation_id], dtype=torch.long)
                        graph.episode_id = torch.tensor([global_episode_id], dtype=torch.long)
                        graph.step_idx = torch.tensor([step_idx], dtype=torch.long)
                        graph.num_drones = torch.tensor([len(active_drones)], dtype=torch.long)
                        graph.obstacles = torch.as_tensor(obstacles, dtype=torch.float32)
                    else:
                        graph = Data(
                            x=x, target=target, edge_index=edge_index, edge_attr=edge_attr,
                            pos=torch.as_tensor(np.asarray(global_positions), dtype=torch.float32),
                            obstacles=torch.as_tensor(obstacles, dtype=torch.float32),
                            formation_id=torch.tensor([formation_id], dtype=torch.long),
                            episode_id=torch.tensor([global_episode_id], dtype=torch.long),
                            step_idx=torch.tensor([step_idx], dtype=torch.long),
                            num_drones=torch.tensor([len(active_drones)], dtype=torch.long),
                        )
                    split_graphs[split_name].append(graph)
                    saved_steps += 1

                # Convergence early-stopping (no frame bypass - just stops the episode)
                if conv_stopping and steps_since_last_event > 50:
                    max_pos_error = 0.0
                    for i, drone_idx in enumerate(active_drones):
                        drone_pos = env.drones[drone_idx].state[3]
                        target_pos = np.array([setpoints[i][0], setpoints[i][1], setpoints[i][3]])
                        error = np.linalg.norm(drone_pos - target_pos)
                        if error > max_pos_error:
                            max_pos_error = error
                    if max_pos_error < conv_threshold:
                        can_stop = True
                        if inject_failures and not failure_injected_this_episode:
                            can_stop = False
                        if dynamic_formation:
                            can_stop = False
                        if can_stop:
                            print(f"      Converged at step {step_idx}. Stopping early.")
                            break

            env.disconnect()

            split_summaries[split_name]["num_graphs"] += saved_steps
            episode_center = np.mean(start_pos[:, :2], axis=0)
            initial_xy_radius = float(np.max(np.linalg.norm(start_pos[:, :2] - episode_center, axis=1)))
            episode_records.append({
                "episode_id": global_episode_id, "split": split_name,
                "split_episode_idx": split_episode_idx, "episode_seed": episode_seed,
                "num_drones": num_drones, "episode_dataset_type": episode_dataset_type,
                "formation_name": formation_name, "initial_xy_limit": xy_limit,
                "initial_xy_radius": initial_xy_radius, "saved_steps": saved_steps,
            })
            global_episode_id += 1

    generated_files = {}
    for split_name, graphs in split_graphs.items():
        if not graphs: continue
        split_dataset_path = f"{dataset_prefix}_{split_name}.pt"
        save_dataset(split_dataset_path, graphs, FORMATION_NAMES, split_name)
        generated_files[split_name] = os.path.basename(split_dataset_path)
        print(f"Generated {split_name} dataset -> {split_dataset_path}")

    metadata_path = f"{dataset_prefix}_metadata.json"
    metadata = {
        "dataset_name": dataset_name, "dataset_type": dataset_type, "task_type": task_type,
        "config": {
            "num_episodes": num_episodes, "max_steps": max_steps, "num_obstacles": num_obstacles,
            "obstacle_radius": obstacle_radius, "inject_failures": inject_failures,
            "dynamic_formation": dynamic_formation, "noisy_sensors": noisy_sensors, "noise_variance": noise_variance,
            "environmental_wind": environmental_wind, "graphical": graphical, "communication_radius": communication_radius,
            "include_formation_in_state": include_formation_in_state, "mixed_formation_types": list(mixed_formation_types),
            "split_ratios": list(split_ratios), "seed": seed, "base_xy_limit": base_xy_limit,
            "altitude_range": list(altitude_range), "validation_spread_scale": validation_spread_scale,
            "test_spread_scale": test_spread_scale, "save_interval": save_interval,
            "conv_stopping": conv_stopping, "conv_threshold": conv_threshold,
        },
        "split_summary": split_summaries, "episodes": episode_records,
    }
    write_dataset_metadata(metadata_path, metadata)
    print(f"Generated dataset metadata -> {metadata_path}")

    return generated_files, metadata_path


## Example Usage

The final cell shows a practical default configuration. Leave it commented if you only want the notebook as documentation, or run it to generate split `.pt` files and the metadata JSON.


In [21]:
generated_files, metadata_path = generate_dataset(
    dataset_name="setpoint_mixed_V5",
    dataset_type="mixed_formations",
    task_type="setpoint_prediction",
    noisy_sensors=False,
    environmental_wind=False,
    dynamic_formation=False,
    inject_failures=False,
    communication_radius=10.0,
    include_formation_in_state=True,
    num_episodes=200,
    max_steps=1200,
    save_interval=10,
    conv_stopping=True,
    conv_threshold=0.2,
    max_obstacles=25,
    seed=12345,
)
print(generated_files)
print(metadata_path)


[setpoint_mixed_V5] Starting train episode 1/160 ...
                             
      Converged at step 1034. Stopping early.
[setpoint_mixed_V5] Starting train episode 2/160 ...
                             
[setpoint_mixed_V5] Starting train episode 3/160 ...
                             
      Converged at step 1010. Stopping early.
[setpoint_mixed_V5] Starting train episode 4/160 ...
                             
      Converged at step 773. Stopping early.
[setpoint_mixed_V5] Starting train episode 5/160 ...
                             
      Converged at step 976. Stopping early.
[setpoint_mixed_V5] Starting train episode 6/160 ...
                             
      Converged at step 758. Stopping early.
[setpoint_mixed_V5] Starting train episode 7/160 ...
                             
      Converged at step 783. Stopping early.
[setpoint_mixed_V5] Starting train episode 8/160 ...
                             
      Converged at step 826. Stopping early.
[setpoint_mixed_V5]

NameError: name 'num_obstacles' is not defined

## Inspect a Generated Split

This section uses a small PyTorch Geometric `InMemoryDataset` wrapper to load one generated split exactly the way a PyG training pipeline would. It reconstructs the dataset, inspects one sample graph, and then shows how PyG batches graphs with a `DataLoader`.


In [ ]:
from pathlib import Path



from torch_geometric.loader import DataLoader





class GeneratedSplitDataset(InMemoryDataset):

    def __init__(self, split_path: str | Path):

        self.split_path = Path(split_path)

        payload = torch.load(self.split_path, weights_only=False)

        self.formation_names = payload.get("formation_names", [])

        self.split_name = payload.get("split_name", "unknown")

        super().__init__(root="")

        self.data = payload["data"]

        self.slices = payload["slices"]





inspect_dataset_name = "noiseless_baseline_mixed_formations"

inspect_split_name = "train"



base_d_drive_path = Path(r"D:\Drone_Swarm_Data")

datasets_dir = base_d_drive_path / dataset_name



split_path = datasets_dir / f"{inspect_dataset_name}_{inspect_split_name}.pt"

metadata_path = datasets_dir / f"{inspect_dataset_name}_metadata.json"



if not split_path.exists():

    raise FileNotFoundError(

        f"Could not find split dataset at {split_path}. Generate the dataset first or update inspect_dataset_name."

    )



dataset = GeneratedSplitDataset(split_path)



print(f"Loaded split file: {split_path.name}")

print(f"Stored split name: {dataset.split_name}")

print(f"Available formation names: {dataset.formation_names}")

print(f"Number of graphs: {len(dataset)}")



sample_graph = dataset[0]

print("\nSample graph summary:")

print(sample_graph)

print(f"x shape: {tuple(sample_graph.x.shape)}")

print(f"target shape: {tuple(sample_graph.target.shape)}")

print(f"y shape: {tuple(sample_graph.y.shape)}")
print(f"edge_index shape: {tuple(sample_graph.edge_index.shape)}")
if hasattr(sample_graph, 'edge_attr') and sample_graph.edge_attr is not None:
    print(f"edge_attr shape: {tuple(sample_graph.edge_attr.shape)}")
print(f"episode_id: {int(sample_graph.episode_id.item())}")

print(f"step_idx: {int(sample_graph.step_idx.item())}")

print(f"num_drones: {int(sample_graph.num_drones.item())}")

print(f"formation_id: {int(sample_graph.formation_id.item())}")



loader = DataLoader(dataset, batch_size=4, shuffle=False)

batch = next(iter(loader))

print("\nPyG batch summary:")

print(batch)

print(f"batched x shape: {tuple(batch.x.shape)}")

print(f"batched edge_index shape: {tuple(batch.edge_index.shape)}")

print(f"batch vector shape: {tuple(batch.batch.shape)}")

print(f"graphs in batch: {batch.num_graphs}")



if metadata_path.exists():

    with open(metadata_path, "r", encoding="ascii") as metadata_file:

        metadata = json.load(metadata_file)



    print("\nMetadata summary:")

    print(f"Metadata file: {metadata_path.name}")

    print(f"Configured dataset type: {metadata['dataset_type']}")

    print(f"Configured split files: {metadata['generated_files']}")

    print(f"Split summary: {metadata['split_summary'][inspect_split_name]}")

    print(f"First episode record: {metadata['episodes'][0]}")

else:

    print(f"\nNo metadata file found at {metadata_path}")

FileNotFoundError: Could not find split dataset at /home/amirbnsl/projects/gnn_drone_project/datasets/noiseless_baseline_mixed_formations_train.pt. Generate the dataset first or update inspect_dataset_name.

: 

: 

: 

## Upload a Dataset to Google Drive

This section is intended for Google Colab. It mounts Google Drive and copies the generated split files and metadata JSON into `MyDrive/dataset/<dataset-name>/`. Update the dataset name if you want to upload a different generated dataset bundle.


In [ ]:
import shutil

from pathlib import Path



upload_dataset_name = "formation_mixed_comm_10m_mixed_formations"

google_drive_root = Path("/content/drive/MyDrive")

google_drive_dataset_dir = google_drive_root / "dataset" / upload_dataset_name



try:

    from google.colab import drive

except ImportError as exc:

    raise RuntimeError(

        "This upload cell is intended to run in Google Colab, where google.colab is available."

    ) from exc



drive.mount("/content/drive", force_remount=False)



local_datasets_dir = Path(r"D:\Drone_Swarm_Data")

google_drive_dataset_dir.mkdir(parents=True, exist_ok=True)



artifacts_to_copy = sorted(local_datasets_dir.glob(f"{upload_dataset_name}*.pt"))

metadata_file = local_datasets_dir / f"{upload_dataset_name}_metadata.json"

if metadata_file.exists():

    artifacts_to_copy.append(metadata_file)



if not artifacts_to_copy:

    raise FileNotFoundError(

        f"No dataset artifacts found for {upload_dataset_name} in {local_datasets_dir}."

    )



copied_files = []

for artifact_path in artifacts_to_copy:

    destination_path = google_drive_dataset_dir / artifact_path.name

    shutil.copy2(artifact_path, destination_path)

    copied_files.append(destination_path)



print(f"Uploaded {len(copied_files)} files to {google_drive_dataset_dir}")

for copied_file in copied_files:

    print(f" - {copied_file.name}")


: 

: 

: 